In [1]:
import pandas as pd
import matplotlib.pyplot as plt
import re
import spacy
from sklearn.model_selection import train_test_split
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.layers import TextVectorization
from collections import Counter
import numpy as np

import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, LSTM, Dense, Dropout, Bidirectional
from tensorflow.keras.callbacks import EarlyStopping

I0000 00:00:1776368725.735970   17161 cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.
I0000 00:00:1776368726.034528   17161 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
I0000 00:00:1776368727.447601   17161 cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.


In [2]:
df_clean = pd.read_csv("./asset/sentiment140_cleaned.csv")

In [3]:
df_clean = df_clean.dropna(subset=['text_nlp']).reset_index(drop=True)

X = df_clean['text_nlp']
y = df_clean['target']

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)
print(f"Train : {len(X_train):,} | Test : {len(X_test):,}")

Train : 1,270,277 | Test : 317,570


In [4]:
VOCAB_SIZE = 50000
MAX_LEN = 40
BATCH_SIZE = 200_000

counter = Counter()
for i in range(0, len(X_train), BATCH_SIZE):
    batch = X_train[i : i + BATCH_SIZE]
    for text in batch:
        counter.update(text.split())
    print(f"Batch {i // BATCH_SIZE + 1} compté")

vocab = ['', '[UNK]'] + [word for word, _ in counter.most_common(VOCAB_SIZE - 2)]

vectorizer = TextVectorization(max_tokens=VOCAB_SIZE, output_sequence_length=MAX_LEN)
vectorizer.set_vocabulary(vocab)
print("Terminé !")

Batch 1 compté
Batch 2 compté
Batch 3 compté
Batch 4 compté
Batch 5 compté
Batch 6 compté
Batch 7 compté
Terminé !


E0000 00:00:1776368768.247938   17161 cuda_platform.cc:52] failed call to cuInit: INTERNAL: CUDA error: Failed call to cuInit: UNKNOWN ERROR (303)


In [5]:
def vectorize_in_batches(data, vectorizer, batch_size):
    results = []
    total = len(data)
    
    for i in range(0, total, batch_size):
        batch = data[i : i + batch_size]
        vectorized = vectorizer(batch)
        results.append(vectorized.numpy())
        print(f"Batch {i // batch_size + 1}/{-(-total // batch_size)} traité — {min(i + batch_size, total):,} / {total:,}")
    
    return np.concatenate(results, axis=0)

X_train_vec = vectorize_in_batches(X_train, vectorizer, BATCH_SIZE)
X_test_vec  = vectorize_in_batches(X_test,  vectorizer, BATCH_SIZE)

print(f"X_train_vec shape : {X_train_vec.shape}")
print(f"X_test_vec shape  : {X_test_vec.shape}")

W0000 00:00:1776368779.049655   17161 cpu_allocator_impl.cc:82] Allocation of 64000000 exceeds 10% of free system memory.


Batch 1/7 traité — 200,000 / 1,270,277


W0000 00:00:1776368779.266273   17161 cpu_allocator_impl.cc:82] Allocation of 64000000 exceeds 10% of free system memory.


Batch 2/7 traité — 400,000 / 1,270,277


W0000 00:00:1776368779.493436   17161 cpu_allocator_impl.cc:82] Allocation of 64000000 exceeds 10% of free system memory.


Batch 3/7 traité — 600,000 / 1,270,277


W0000 00:00:1776368779.716900   17161 cpu_allocator_impl.cc:82] Allocation of 64000000 exceeds 10% of free system memory.


Batch 4/7 traité — 800,000 / 1,270,277


W0000 00:00:1776368779.953652   17161 cpu_allocator_impl.cc:82] Allocation of 64000000 exceeds 10% of free system memory.


Batch 5/7 traité — 1,000,000 / 1,270,277
Batch 6/7 traité — 1,200,000 / 1,270,277
Batch 7/7 traité — 1,270,277 / 1,270,277
Batch 1/2 traité — 200,000 / 317,570
Batch 2/2 traité — 317,570 / 317,570
X_train_vec shape : (1270277, 40)
X_test_vec shape  : (317570, 40)


In [6]:
EMBEDDING_DIM = 64   # dimension des vecteurs d'embedding

model = Sequential([
    # Couche Embedding : transforme chaque token (entier) en vecteur dense
    Embedding(input_dim=VOCAB_SIZE, output_dim=EMBEDDING_DIM),

    # LTSM Bidirectionnel : lit la séquence dans les deux sens
    Bidirectional(LSTM(64, return_sequences=False)),

    # Dropout : désactive aléatoirement 40% des neurones → évite l'overfitting
    Dropout(0.4),

    # Couche Dense intérmediaire
    Dense(32, activation='relu'),

    Dropout(0.2),
    
    # Couche de sortie : 1 neurone, sigmoid → probabilité entre 0 et 1
    Dense(1, activation='sigmoid')
])
model.compile(
    optimizer='adam',
    loss='binary_crossentropy',  # classification binaire
    metrics=['accuracy']
)
model.build(input_shape=(None, MAX_LEN))
model.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding (Embedding)           │ (None, 40, 64)         │     3,200,000 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ bidirectional (Bidirectional)   │ (None, 128)            │        66,048 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 32)             │         4,128 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 32)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 1)              │            33 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 3,270,209 (12.47 MB)

 Trainable params: 3,270,209 (12.47 MB)

 Non-trainable params: 0 (0.00 B)

In [7]:
early_stop = EarlyStopping(monitor='val_loss', patience=4, restore_best_weights=True)

model.fit(
    X_train_vec, y_train,
    shuffle=True, 
    epochs=30,
    validation_split=0.1,
    batch_size=512,
    callbacks=[early_stop],
    verbose=1
)

Epoch 1/30
2233/2233 ━━━━━━━━━━━━━━━━━━━━ 195s 87ms/step - accuracy: 0.7680 - loss: 0.4845 - val_accuracy: 0.7819 - val_loss: 0.4596
Epoch 2/30
2233/2233 ━━━━━━━━━━━━━━━━━━━━ 227s 102ms/step - accuracy: 0.7919 - loss: 0.4460 - val_accuracy: 0.7834 - val_loss: 0.4583
Epoch 3/30
2233/2233 ━━━━━━━━━━━━━━━━━━━━ 225s 101ms/step - accuracy: 0.8027 - loss: 0.4261 - val_accuracy: 0.7832 - val_loss: 0.4638
Epoch 4/30
2233/2233 ━━━━━━━━━━━━━━━━━━━━ 220s 99ms/step - accuracy: 0.8130 - loss: 0.4054 - val_accuracy: 0.7810 - val_loss: 0.4813
Epoch 5/30
2233/2233 ━━━━━━━━━━━━━━━━━━━━ 219s 98ms/step - accuracy: 0.8228 - loss: 0.3850 - val_accuracy: 0.7789 - val_loss: 0.5062
Epoch 6/30
2233/2233 ━━━━━━━━━━━━━━━━━━━━ 220s 98ms/step - accuracy: 0.8316 - loss: 0.3660 - val_accuracy: 0.7747 - val_loss: 0.5403


In [8]:
# Evaluation sur le jeu de test
loss, accuracy = model.evaluate(X_test_vec, y_test, verbose=1)

print(f"Loss     : {loss:.4f}")
print(f"Accuracy : {accuracy:.4f} ({accuracy*100:.2f}%)")

9925/9925 ━━━━━━━━━━━━━━━━━━━━ 31s 3ms/step - accuracy: 0.7833 - loss: 0.4582
Loss     : 0.4582
Accuracy : 0.7833 (78.33%)
